# 02 — Literature & Scientific Evidence

**CuSelect-AI · Chapter II — From a falsifiable protocol to attributable evidence**  
**Campaign:** `CU-001` · **Scientific focus:** Cu-based catalysts and CO₂ reduction to ethylene · **Notebook status:** Evidence collection and verification workflow

**Previously established:** Notebook 00 verified the computational foundation and initial *unrelaxed* Cu/Cu–Ag geometries. Notebook 01 defined a falsifiable research question, outcome metrics, baselines and publication-level holdout policy. Neither notebook measured catalytic performance or validated a mechanistic claim.

**The question now:** *Which published claims can we identify, locate in original sources, distinguish by evidence type, and validate sufficiently to support later data-science and AI decisions?*

**Inputs**: `configs/CU-001_protocol.json`, `data/papers.csv` (eight **unverified seeds**), `src/cuselect/evidence.py` and source publications opened by a human reviewer.  
**Outputs**: bibliographic audit CSV, claim-review queue CSV, draft-claim CSV, an initially header-only verified-claims CSV, and a readiness report.  
**Acceptance gate**: No claim becomes *human_verified* merely because its DOI resolves, an abstract mentions it, or an LLM repeats it.

> **Research integrity:** This notebook does **not** download copyrighted PDFs automatically, predict ethylene FE, construct DFT reaction barriers, or claim the hypothesis has been demonstrated. Online metadata lookup is optional. Every scientific claim needs its precise source location and human verification before use as ground truth.

## 2.1 — Recover the approved research context

The literature search must start from the question we already defined, not from arbitrary interesting papers. Our protocol establishes **what counts as an error** and prevents us from changing evaluation criteria after inspecting results.

**Scientific boundary:** `CU-001-AG1` is a periodic, unrelaxed Cu(100) substitution model; the experimental paper on Ag-modified Cu **nanowires** is a different catalyst system. Its results may motivate a hypothesis, but they cannot automatically be assigned to our slab.

In [ ]:
# PSEUDOCODE
# 1. Locate the project root regardless of VS Code's working directory.
# 2. Load the approved study protocol and eight seed bibliography records.
# 3. State the hypothesis and verify that evaluation is not already declared complete.

from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from cuselect.project_checks import require_project_root
from cuselect.protocol import load_research_protocol
from cuselect.evidence import (normalize_doi, audit_seed_bibliography,
    verify_metadata, EvidenceClaim, check_claim_links, claims_frame, save_csv)

start = Path.cwd().resolve()
for candidate_root in (start, *start.parents):
    if (candidate_root / 'pyproject.toml').is_file():
        ROOT = require_project_root(candidate_root)
        break
else:
    raise RuntimeError('Project root not found; open a notebook inside CuSelect-AI.')

protocol = load_research_protocol(ROOT / 'configs' / 'CU-001_protocol.json')
papers = pd.read_csv(ROOT / 'data' / 'papers.csv', dtype={'paper_id':'string','doi':'string'})
assert protocol.campaign_id == 'CU-001'
assert protocol.evaluation.evaluation_status == 'planned_not_run'
assert len(papers) == 8
print('Project:', ROOT.name)
print('Research campaign:', protocol.campaign_id)
print('Target:', protocol.system.target_product)
print('Seed bibliography:', len(papers), 'entries; not yet claim-verified')

## 2.2 — Bibliographic sanity checks: identifiers are not evidence

A **Digital Object Identifier (DOI)** identifies a registered scholarly object. Before reading a paper, check the DOI's syntax, duplicates, and expected publication year. Later we compare its title and year with publisher-deposited metadata.

**Important levels:** (i) valid-looking DOI, (ii) matching publication metadata, (iii) article and licensing reviewed, (iv) *specific scientific claim checked against a source location*. Only the fourth level can support a verified scientific claim.

In [ ]:
# PSEUDOCODE
# 1. Normalize every DOI; verify only syntax and duplicates.
# 2. Display problems; preserve the original bibliographic seeds.
# 3. Fail early if the seed list itself contains invalid identifiers.

seed_audit = audit_seed_bibliography(papers)
display(seed_audit[['paper_id','doi','doi_syntax_valid','duplicate_doi','publication_year_valid','bibliographic_status']])
assert seed_audit['doi_syntax_valid'].all(), 'At least one seed DOI has invalid syntax.'
assert not seed_audit['duplicate_doi'].any(), 'Duplicate DOIs require review.'
assert not seed_audit['duplicate_paper_id'].any(), 'Duplicate internal paper IDs require review.'
assert seed_audit['publication_year_valid'].all(), 'Publication years require manual correction.'
print('DOI syntax and duplicate checks passed. Scientific claims are NOT verified.')

## 2.3 — Optional publisher-deposited metadata retrieval

**Why Crossref?** Crossref provides a public REST API for scholarly metadata, including journal, title, DOI, and publication dates. Its metadata is deposited by publishers and other members. Metadata confirmation does **not** validate a paper's experiments, DFT calculations, or mechanisms.

- By default, this revised notebook runs **offline**. If a compatible prior Crossref metadata audit exists in `data/interim/02_metadata_audit.csv`, it can be reused and clearly identified as cached data. Otherwise records are labeled `not_queried`.
- When you have an internet connection on your Mac, change `RUN_LIVE_CROSSREF_LOOKUP` to `True`. Each request has a timeout; failures are recorded rather than silently treated as success.
- A title/year discrepancy triggers human review. Differences between publication-online and issue year can be legitimate.
- Copyright and licensing require a separate check at the article level; Crossref's presence of license metadata is not a full license review.

Official documentation: https://www.crossref.org/documentation/retrieve-metadata/rest-api/
- The cache is a record of a **previous lookup**, not evidence that Crossref was contacted in the current run. Neither a cache hit nor a fresh metadata match verifies scientific claims.


In [ ]:
# PSEUDOCODE
# 1. Decide whether to query Crossref again or reuse a previously saved audit.
# 2. If using cached metadata, check that every paper ID and DOI still matches our seeds.
# 3. Never equate bibliographic metadata with scientific claim verification.

RUN_LIVE_CROSSREF_LOOKUP = False  # Set True to perform a NEW Crossref lookup on your Mac.
REUSE_PREVIOUS_METADATA_AUDIT = True
prior_audit_path = ROOT / 'data' / 'interim' / '02_metadata_audit.csv'

if RUN_LIVE_CROSSREF_LOOKUP:
    metadata_audit = verify_metadata(papers, online=True)
    metadata_provenance = 'live_crossref_lookup_this_run'
elif REUSE_PREVIOUS_METADATA_AUDIT and prior_audit_path.exists():
    previous = pd.read_csv(prior_audit_path, dtype={'paper_id':'string', 'doi':'string'})
    seed_pairs = {(str(x.paper_id), normalize_doi(str(x.doi))) for x in papers.itertuples(index=False)}
    prior_pairs = {(str(x.paper_id), normalize_doi(str(x.doi))) for x in previous.itertuples(index=False)}
    required = {'metadata_status','title_similarity','year_consistent','claim_review_status'}
    if seed_pairs != prior_pairs or not required.issubset(previous.columns) or len(previous) != len(papers):
        raise ValueError('Previous metadata audit does not match current bibliography; rerun Crossref lookup.')
    metadata_audit = previous.copy()
    metadata_provenance = ('previous_offline_audit_reused_not_queried'
                           if previous['metadata_status'].eq('not_queried').all()
                           else 'previous_metadata_audit_reused_not_requeried')
else:
    metadata_audit = verify_metadata(papers, online=False)
    metadata_provenance = 'not_queried'

metadata_cols = ['paper_id','doi','metadata_status','title_similarity','year_consistent','claim_review_status']
display(metadata_audit[metadata_cols])
print('Metadata provenance:', metadata_provenance)
print('Metadata review states:', metadata_audit['metadata_status'].value_counts().to_dict())
print('A DOI/title/year match does NOT verify a scientific claim.')

## 2.3b — A reproducible literature search, not a collection of convenient papers

The eight seeds are our *starting sample*, not a systematic review. Before expanding the evidence base, define the search objective, queries, databases, eligibility criteria, date and deduplication policy.

**Discovery sources:** Crossref and OpenAlex for broad bibliographic metadata; Semantic Scholar for scholarly search; publisher pages, institutional access, and discipline databases for full-text review. Scopus and Web of Science may require university subscriptions. Preprints must be labeled as such and not treated as equivalent to peer-reviewed articles.

**Inclusion priorities:** Cu or Cu-alloy CO₂RR studies with ethylene/ethanol mechanisms; agentic catalyst-discovery methods with clear baselines; applicable atomistic ML and interface models; authoritative catalyst datasets. **Exclude or separately label:** unrelated chemistries, unsupported claims, retracted papers, duplicate records and findings lacking extractable conditions.

The next cell records a **search plan only**. It does not secretly query external search engines. When searches are actually performed, record the search date, engine, result count, inclusions/exclusions and reasons, and append newly confirmed records to a reviewed bibliography—not directly to the evidence ledger.

In [ ]:
# PSEUDOCODE
# 1. Predefine searches across chemistry, electrochemistry, materials and AI.
# 2. Record scope and review status before actually executing the searches.
# 3. Save a transparent plan; do not pretend planned searches have been run.

search_plan = pd.DataFrame([
    {'research_stream':'CO2RR Cu-Ag mechanisms','search_query':'("CO2 reduction" OR CO2RR) AND (Cu Ag OR copper silver) AND (ethylene OR ethanol) AND (mechanism OR coupling)'},
    {'research_stream':'Cu single-atom alloys','search_query':'copper single-atom alloy CO2 electroreduction C2+ selectivity'},
    {'research_stream':'AI catalyst workflows','search_query':'multi-agent LLM catalyst discovery DFT machine learning'},
    {'research_stream':'Electrode interface ML','search_query':'MLIP copper water interface constant potential electrochemistry'},
    {'research_stream':'Electrochemical datasets','search_query':'CO2 reduction catalyst dataset experimental conditions faradaic efficiency'},
    {'research_stream':'Scientific verification','search_query':'verification first autonomous catalysis evidence uncertainty LLM'},
])
search_plan['suggested_sources'] = 'Crossref;OpenAlex;Semantic Scholar;publisher databases'
search_plan['search_status'] = 'planned_not_executed'
search_plan['searched_on'] = ''
search_plan['hits_found'] = ''
search_plan['screening_notes'] = ''
display(search_plan[['research_stream','search_query','search_status']])
assert (search_plan['search_status']=='planned_not_executed').all()


## 2.4 — Read the papers according to scientific function

Our eight publications are not eight independent answers to the same chemistry question. Some discuss **CO₂RR experiments**, some describe **autonomous-agent architectures**, one is a **dataset paper**, and one considers **constant-potential molecular modeling**.

| Paper | First question for the reviewer | Key distinction to preserve |
|:--|:--|:--|
| P001 | What exactly is the Cu-alloy agent optimizing? | C₂⁺ is not necessarily C₂H₄ alone |
| P002 | What was autonomously optimized and against which baseline? | Agent performance does not prove experimental selectivity |
| P003 | Which chemistry does MAESTRO actually study? | Its published case is **ORR**, not Cu CO₂RR |
| P004 | Is the contribution experimental or conceptual? | A **Perspective** outlining verification-first catalysis |
| P005 | How do Ag nanoparticles and Ag single atoms affect the product ratio? | Ag-modified **nanowires** are not the 3×3 Cu(100) slab |
| P006 | What data and label quality exist in the CO₂RR corpus? | Extracted corpus fields are not automatically ground truth |
| P007 | What does Al/Zn dual-doping improve, and under what conditions? | Reported **C₂⁺** FE must not be relabeled ethylene FE |
| P008 | How is electrochemical potential represented? | Generic energy/force MLIPs are not constant-potential models |

This matrix tells us *what to inspect*; it is not an expert-verification certificate. Begin with publisher abstracts, methods, figures, supplements, and accessible full text. For each claim, identify the exact section, figure, table, or supplementary reference.

In [ ]:
# PSEUDOCODE
# 1. Prepare one specific verification task per seed publication.
# 2. Link the task to the original paper ID and DOI.
# 3. Never mark a task as completed merely because the title was retrieved.

review_questions = {
    'P001':'Separate C2+ design claims from ethylene-specific evidence; identify DFT and experimental support.',
    'P002':'Extract the task, simulation budget, baseline, and evidence supporting claimed savings.',
    'P003':'Confirm the ORR task and isolate transferable software methods from nontransferable chemistry.',
    'P004':'Record recommended verification practices; label these as perspective/methodology claims.',
    'P005':'Check Ag nanoparticles vs Ag single atoms, ethanol/ethylene ratio, reactor and conditions.',
    'P006':'Locate the annotated CO2RR dataset, available downloads, label provenance and licenses.',
    'P007':'Differentiate C2+ FE from ethylene FE; identify dual dopants and flow-cell conditions.',
    'P008':'Inspect potential-control method and Cu/water model transferability claims.',
}
review_queue = papers[['paper_id','doi','kind']].copy()
review_queue['doi'] = review_queue['doi'].map(normalize_doi)
review_queue['article_url'] = 'https://doi.org/' + review_queue['doi']
review_queue['first_review_question'] = review_queue['paper_id'].map(review_questions)
review_queue['full_text_access'] = 'not_assessed'
review_queue['article_license'] = 'not_assessed'
review_queue['source_review_status'] = 'pending_human_review'
assert review_queue['first_review_question'].notna().all()
display(review_queue[['paper_id','kind','first_review_question','source_review_status']])

## 2.5 — A scientific claim is not a bibliographic record

We now distinguish five frequent kinds of scientific statements:

| Claim class | Typical example | What would justify the claim? |
|---|---|---|
| `experimental_observation` | A product ratio changed between two catalyst preparations | Methods, original measured data, uncertainty and conditions |
| `computed_result` | A modeled intermediate has a particular adsorption energy | Exact structures, calculation reference, model, convergence and units |
| `author_interpretation` | A mechanistic pathway is *proposed* to explain a trend | The authors' discussion plus supporting or competing evidence |
| `hypothesis` | A new substitution pattern *might* favor ethylene | Marked as untested until an appropriate validation |
| `methodology` / `review_context` | A multi-agent workflow or verification policy was proposed | Accurate characterization of that paper's scope |

**Never confuse** ethylene FE with ethanol/ethylene ratios, C₂⁺ FE, C₂⁺ partial current density, or CO adsorption energy. Never transfer a result between different electrodes, reactors, electrolytes, or potentials without explicit applicability assessment.

A claim may use an `Abstract` location initially, but that gives **abstract-level support only**. More specific mechanistic or numerical assertions may require figures, methods, tables and supplements.

## 2.6 — Create a *draft* evidence ledger (not a verified dataset)

To practice rigorous extraction, the next cell prepares a handful of **paraphrased review candidates** based on published abstracts. They are clearly marked `pending_source_review` and are **not** ground truth. We deliberately leave the numerical FE column empty: the Ag–Cu ratios are *product ratios*, not Faradaic efficiencies. A human must independently inspect the original paper and document the exact source location before changing any draft to `human_verified`.

**Review workflow:** open paper → inspect passage/figure → record exact location and conditions → check claim against source → annotate limitations → separately approve or reject. We store the reusable schema and validation rules in `src/cuselect/evidence.py`.

In [ ]:
# PSEUDOCODE
# 1. Create a small collection of literature-guided questions as DRAFT claims.
# 2. Validate the schema, DOI linkage, split group and data ranges.
# 3. Keep review_status pending; nothing is promoted by code execution.

claim_drafts = [
    EvidenceClaim(
        evidence_id='E-DRAFT-001', paper_id='P005', doi='10.1038/s41467-024-54636-w',
        claim_paraphrase='Wang et al. report ethanol-to-ethylene product ratios of 0.41 for AgCu NW (Ag nanoparticles) and 4.26 for Ag1Cu NW (single Ag atoms).',
        claim_type='experimental_observation', source_extent='abstract', exact_source_location='Abstract, sentence reporting ethanol-to-ethylene ratios (independent reviewer check pending)',
        catalyst_system='AgCu NW (Ag nanoparticles) and Ag1Cu NW (single Ag atoms) on Cu nanowires; NOT our Cu(100) slab',
        target_or_observation='ethanol/ethylene product ratio; NOT ethylene FE',
        attribution_note='Publisher abstract reports product ratios, NOT ethylene FE; verify exact source independently before promotion.'),
    EvidenceClaim(
        evidence_id='E-DRAFT-002', paper_id='P003', doi='10.1021/acscentsci.6c01260',
        claim_paraphrase='MAESTRO describes a multi-agent catalyst design workflow evaluated for oxygen reduction rather than Cu-based CO2 reduction.',
        claim_type='methodology', source_extent='abstract', exact_source_location='Abstract (confirm independently)',
        target_or_observation='agent workflow methodology, ORR chemistry',
        attribution_note='Transferrable architecture is not direct Cu CO2RR performance evidence.'),
    EvidenceClaim(
        evidence_id='E-DRAFT-003', paper_id='P004', doi='10.1038/s44387-026-00111-4',
        claim_paraphrase='A perspective advocates verification-first autonomous catalysis, emphasizing evidence records and uncertainty-aware validation.',
        claim_type='review_context', source_extent='abstract', exact_source_location='Abstract (confirm independently)',
        target_or_observation='methodological guidance, not validation of CuSelect-AI',
        attribution_note='Perspective paper; do not label as CuSelect-AI experimental proof.'),
    EvidenceClaim(
        evidence_id='E-DRAFT-004', paper_id='P007', doi='10.1002/anie.5598898',
        claim_paraphrase='A dual-dopant Cu alloy study reports enhanced C2+ formation under specified flow-cell conditions.',
        claim_type='experimental_observation', source_extent='abstract', exact_source_location='Abstract (confirm independently)',
        target_or_observation='C2+ products; do not infer ethylene-specific FE',
        attribution_note='Review product definitions and conditions in full text.'),
]
check_claim_links(claim_drafts, papers)
drafts_df = claims_frame(claim_drafts)
display(drafts_df[['evidence_id','paper_id','claim_type','claim_paraphrase','review_status']])
assert drafts_df['review_status'].eq('pending_source_review').all()
print('Draft review candidates:', len(drafts_df), '| Human-verified claims: 0')

## 2.7 — Quality control, applicability, and contradictions

**Validation is multidimensional:** source authenticity, exact source location, evidence class, units, experimental context, chemical applicability and publication-level data leakage. A difference in reported product selectivity is **not automatically a contradiction** when active sites, reactor setups, potentials or product definitions differ.

Before a claim can support an AI decision, ask:

1. **Provenance:** Where precisely was the result reported? Abstract, page, figure, table or supplement?
2. **Evidence:** Is the statement measured, computed, interpreted, hypothesized, or a methodological opinion?
3. **Conditions:** Which catalyst architecture, current density, potential reference, electrolyte, cell and normalization?
4. **Transferability:** Is this comparable with the Cu(100) slab? Why or why not?
5. **Leakage:** Was this publication assigned to development or withheld evaluation? Can our RAG agent access it?

For our first pilot, P005 can guide early **development**, but if its findings are in the agent's retrieval corpus, **P005 cannot also serve as an unseen holdout paper**. Split at DOI/publication level and enforce access controls later.

In [ ]:
# PSEUDOCODE
# 1. Check that draft claims have valid paper links and remain unapproved.
# 2. Confirm distinct claim types and publication-level holdout keys.
# 3. Prepare an explicit set of scientific gaps to investigate.

assert drafts_df['evidence_id'].is_unique
assert drafts_df['split_group'].eq(drafts_df['doi']).all()
assert drafts_df['faradaic_efficiency_percent'].isna().all(), 'Do not encode product ratios as FE.'

scientific_gaps = pd.DataFrame([
    {'topic':'Cu-Ag applicability', 'missing_evidence':'Surface morphology and operating conditions that make nanowire results transferable to a Cu(100) slab', 'status':'open'},
    {'topic':'Mechanism discrimination', 'missing_evidence':'Competing ethylene/ethanol pathways under comparable potential and surface conditions', 'status':'open'},
    {'topic':'Model validity', 'missing_evidence':'Independent atomistic benchmark and uncertainty for selected Cu-Ag configurations', 'status':'open'},
    {'topic':'Agent novelty', 'missing_evidence':'Held-out tests showing reduced false advancements without excessive false rejection', 'status':'open'},
])
display(scientific_gaps)

## 2.7a — P005 guided source review: a real experimental case, not an automatic verification

**Primary source:** Wang et al., *Nature Communications* **15**, 10247 (2024), DOI [10.1038/s41467-024-54636-w](https://doi.org/10.1038/s41467-024-54636-w).

**Source-grounded candidate finding (Abstract):** the authors report ethanol-to-ethylene product ratios of **0.41 for AgCu NW** (Ag nanoparticles decorating Cu nanowires) and **4.26 for Ag₁Cu NW** (single Ag atoms on Cu nanowires). **These numbers are product ratios, not Faradaic efficiencies.** The ratio's exact experimental conditions cannot be inferred from the abstract alone; do not fill them from unrelated measurements.

The full paper separately describes flow-cell maxima of **54.9% ethylene FE** for AgCu NW at **−1.1 V vs RHE**, and **56.3% ethanol FE** for Ag₁Cu NW at **−1.0 V vs RHE** (1 M KOH, as reported). These are **different observations** with different target products and must receive different evidence IDs if entered. The reported potentials were **not iR-corrected**. The quoted current densities are not to be relabeled as ethylene/ethanol *partial* current densities without checking the definition in the paper. Review **Figure 2, its caption, Results, and the cited Supplementary Figures** before extracting data. [Original publisher article](https://www.nature.com/articles/s41467-024-54636-w).

**Mechanistic distinction:** the authors propose different coupling routes and support them with operando spectroscopy, isotope experiments and DFT. Their mechanistic interpretation is **not** a direct experimental measurement of every elementary step. If entered, classify the proposed route as `author_interpretation` rather than `experimental_observation`.

**Model applicability:** none of these nanowire FE values is a prediction for our idealized, unrelaxed, periodically repeated Cu(100)/Ag slab. P005 is a **development example**, not an unseen benchmark test article.

**Reviewer checklist:** confirm the precise abstract sentence, identify each catalyst configuration, record the source location, review Fig. 2 and supplementary information, note conditions and limitations, and only then determine whether to explicitly mark `human_verified` in the separate review CSV. Code execution is **not** the human review.


In [ ]:
# PSEUDOCODE
# 1. Open the existing human-review file from its *correct* location.
# 2. Inspect the P005 draft and any existing human-review entry.
# 3. Keep all statuses unchanged: inspecting a paper does not verify it.

review_path = ROOT / 'data' / 'references' / '02_human_reviewed_claims.csv'
expected_columns = list(EvidenceClaim.model_fields)

if review_path.exists():
    review_table = pd.read_csv(review_path, dtype=str, keep_default_na=False)
    missing_cols = sorted(set(expected_columns) - set(review_table.columns))
    extra_cols = sorted(set(review_table.columns) - set(expected_columns))
    if missing_cols or extra_cols:
        raise ValueError(f'Human-review schema mismatch. Missing: {missing_cols}; extra: {extra_cols}')
else:
    review_table = pd.DataFrame(columns=expected_columns)
    print('Review file not found yet. Create it from the header template before human approval.')

p005_draft = drafts_df.loc[drafts_df['paper_id'].eq('P005')].copy()
p005_existing = review_table.loc[review_table['paper_id'].eq('P005')].copy()

print('P005 draft evidence records:', len(p005_draft))
print('P005 human-review file records:', len(p005_existing))
print('Review file:', review_path.relative_to(ROOT))
display(p005_draft[['evidence_id','claim_paraphrase','claim_type','review_status']])
if not p005_existing.empty:
    display(p005_existing[['evidence_id','exact_source_location','source_checked_by_human','review_status','reviewer_note']])

assert drafts_df['review_status'].eq('pending_source_review').all()
print('No scientific claims were promoted by this cell.')

## 2.7b — How to approve evidence without fabricating verification

The file `data/references/02_human_reviewed_claims.csv` is the **reviewer's input**, not a generated output. If you have already added `E-DRAFT-001` there, **keep that record**—do not overwrite it or create a duplicate. A correct pending row normally has:

- `claim_type = experimental_observation`; `source_extent = abstract`; `exact_source_location = Abstract, sentence describing the 0.41 → 4.26 product-ratio comparison`;
- `conditions`, `faradaic_efficiency_percent` and `fe_product` **blank for the product-ratio claim**, unless the specific source statement provides compatible information;
- `source_checked_by_human = False`, `review_status = pending_source_review` **until the original source is independently checked**;
- `split_group = 10.1038/s41467-024-54636-w`, so all results from this paper remain in the same data partition.

When you have personally checked the original source, edit the row **in the CSV**: use `source_checked_by_human=True`, `review_status=human_verified` (not `verified`), and a meaningful `reviewer_note` describing exactly what was checked. Make sure the evidence statement is consistent with its source. Then rerun Section 2.8: the Pydantic validator and DOI-link checks must pass. **Never** write `source_checked_by_human = True` as a free-standing notebook variable; that neither proves source verification nor updates the evidence database.


## 2.8 — Human review gate and reproducible export

The program will export three **different** things:

1. **Metadata audit**: Which bibliographic entries have been checked with Crossref? Online lookup is optional.
2. **Draft-claim queue**: What claims need source review? They remain *pending* and cannot be treated as verified.
3. **Verified-claim dataset**: Only entries explicitly approved by a human from the dedicated review file. Initially this file contains **column headers and zero rows**.

The optional human-review input lives at `data/references/02_human_reviewed_claims.csv`. Fill it only after reading the source; use the column schema from the draft file. Do **not** edit the generated draft queue and expect your changes to persist, because rerunning the notebook regenerates it.

If you provide a reviewed file, the cell validates it; a `human_verified` row needs `source_checked_by_human=True`, a precise location and a reviewer note. **This enforces documentation, but is not a substitute for independent expert assessment.**

In [ ]:
# PSEUDOCODE
# 1. Save metadata and pending review queues.
# 2. Read separately submitted human-reviewed claims, if any.
# 3. Validate source links and only export explicitly human-verified entries.
# 4. Produce a status report with honest readiness flags.

metadata_path = save_csv(metadata_audit, ROOT / 'data' / 'interim' / '02_metadata_audit.csv')
queue_path = save_csv(review_queue, ROOT / 'data' / 'interim' / '02_source_review_queue.csv')
search_plan_path = save_csv(search_plan, ROOT / 'data' / 'interim' / '02_search_plan.csv')
draft_path = save_csv(drafts_df, ROOT / 'data' / 'interim' / '02_draft_claims.csv')

human_review_path = ROOT / 'data' / 'references' / '02_human_reviewed_claims.csv'
reviewed_claims = []
if human_review_path.exists():
    human_table = pd.read_csv(human_review_path, dtype=str, keep_default_na=False)
    for row in human_table.to_dict('records'):
        if not str(row.get('evidence_id','')).strip():
            continue
        if row.get('faradaic_efficiency_percent') == '':
            row['faradaic_efficiency_percent'] = None
        reviewed_claims.append(EvidenceClaim.model_validate(row))

check_claim_links(reviewed_claims, papers)
verified_claims = [c for c in reviewed_claims if c.review_status == 'human_verified']
verified_df = claims_frame(verified_claims)
verified_path = save_csv(verified_df, ROOT / 'data' / 'processed' / '02_verified_claims.csv')

report = {
    'campaign_id': protocol.campaign_id,
    'notebook': '02_Literature_and_Evidence',
    'seed_papers': int(len(papers)),
    'metadata_online_lookup_enabled': RUN_LIVE_CROSSREF_LOOKUP,
    'metadata_provenance': metadata_provenance,
    'metadata_status_counts': {str(k):int(v) for k,v in metadata_audit['metadata_status'].value_counts().items()},
    'draft_claims': len(claim_drafts),
    'planned_searches_not_executed': len(search_plan),
    'human_verified_claims': len(verified_claims),
    'unresolved_scientific_gaps': len(scientific_gaps),
    'evidence_ready_for_rag': len(verified_claims) > 0,  # limited, source-traceable retrieval only; not a comprehensive corpus
    'ready_for_blinded_benchmark': False,  # planned search and independent benchmarking have not been completed
    'evaluation_status': protocol.evaluation.evaluation_status,
    'scientific_selectivity_predictions_available': False,
    'next_notebook': '03_Data_Audit_and_Curation',
}
report_dir = ROOT / 'outputs' / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / '02_evidence_readiness.json'
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')

print('Metadata audit:', metadata_path.relative_to(ROOT))
print('Source review queue:', queue_path.relative_to(ROOT))
print('Search plan:', search_plan_path.relative_to(ROOT))
print('Draft claims:', draft_path.relative_to(ROOT))
print('Verified claims:', verified_path.relative_to(ROOT))
print('Readiness report:', report_path.relative_to(ROOT))
print('Human-verified claims:', report['human_verified_claims'])
print('Ready for evidence-grounded RAG:', report['evidence_ready_for_rag'])
print('Ready for blinded scientific benchmarking:', report['ready_for_blinded_benchmark'])


## 2.9 — Interpret what we have (and have not) established

**Established by executing this notebook:** We can load the protocol, audit DOI syntax/duplicates, optionally compare bibliographic metadata with Crossref, classify literature questions, validate draft-claim schemas, and export a provenance-aware review queue. These are **research-infrastructure accomplishments**, not new electrochemical findings.

**Not established:** We have **not automatically verified full-text claims**, reproduced catalyst performance, calculated reaction barriers, established ethylene FE, or demonstrated the scientific contribution of the critic. The remaining gap is human-reviewed, source-located evidence and independently validated observations.

**Scientific review before Notebook 03:** Open the source-review queue; verify publication metadata and access rights; inspect the relevant figures/tables/supplements; populate `data/references/02_human_reviewed_claims.csv` with genuine, exactly located claims. Rerun Sections 2.7–2.8. If the verified-claims table is empty, Notebook 03 may inspect data schemas but **must not proceed as though a labeled scientific dataset exists**.

**Handoff to Notebook 03 — Data Audit & Curation:**

- **Inputs:** `data/interim/02_metadata_audit.csv`, `data/interim/02_source_review_queue.csv`, `data/interim/02_search_plan.csv`, `data/processed/02_verified_claims.csv` and `outputs/reports/02_evidence_readiness.json`.
- **Question:** Which independently verified observations are actually comparable across catalyst geometry, electrolyte, potential reference, cell configuration and product definition?
- **Transition gate:** Restart kernel → Run All → `python -m pytest -q` → inspect outputs → confirm nonempty verified evidence **before any supervised training or mechanistic benchmark**.

> **Next scientific action:** Begin with P005 (Ag-modified Cu nanowires) for the qualitative development case; independently verify its measurements and mechanism interpretations. Keep any future blinded holdout publication entirely inaccessible to the agent's retrieval corpus.
> **Correction applied:** Temporary file-search/debugging cells and a free-standing `review_status = 'verified'` assignment were removed. They were not evidence verification and must not be used to promote records. The only accepted approved status is `human_verified` after genuine source review.
